In [ ]:
!pip install dm-sonnet -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.4/268.4 kB 7.0 MB/s eta 0:00:00


In [ ]:
import re
import nltk
import numpy as np
import pandas as pd
import sonnet as snt
import tensorflow as tf

from nltk.corpus import stopwords

nltk.download('stopwords')

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
df = pd.read_csv('/content/Corona.csv', encoding='latin-1')
df.head()

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
3,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive
4,3803,48755,NaN,16-03-2020,"Me, ready to go at supermarket during the #COV...",Extremely Negative


In [ ]:
df = df[['OriginalTweet']][:5000].dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 1 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   OriginalTweet  5000 non-null   object
dtypes: object(1)
memory usage: 39.2+ KB


In [ ]:
stop_words = stopwords.words('english')

def data_cleaner(tweet):
    tweet = re.sub(r'http\S+', ' ', tweet)
    tweet = re.sub(r'<.*?>',' ', tweet)
    tweet = re.sub(r'\d+',' ', tweet)
    tweet = re.sub(r'#\w+',' ', tweet)
    tweet = re.sub(r'@\w+',' ', tweet)
    tweet = tweet.split()
    tweet = " ".join([word for word in tweet if not word in stop_words])
    return tweet

df['OriginalTweet'] = df['OriginalTweet'].apply(data_cleaner)
df.head()

,OriginalTweet
0,
1,advice Talk neighbours family exchange phone n...
2,Coronavirus Australia: Woolworths give elderly...
3,"My food stock one empty... PLEASE, panic, THER..."
4,"Me, ready go supermarket outbreak. Not I'm par..."


In [ ]:
tokenizer = Tokenizer()

tokenizer.fit_on_texts(df['OriginalTweet'])

sequences = tokenizer.texts_to_sequences(df['OriginalTweet'])

X = []
y = []
for seq in sequences:
    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

##### Создание модели

In [ ]:
class RNN(snt.RNNCore):

  def __init__(self, hidden_size, activation=tf.tanh, name="vanilla_rnn"):
    super(RNN, self).__init__(name=name)
    self._hidden_size = hidden_size
    self._activation = activation

  def _build(self, input_, prev_state):
    self._in_to_hidden_linear = snt.Linear(
        self._hidden_size, name="in_to_hidden")

    self._hidden_to_hidden_linear = snt.Linear(
        self._hidden_size, name="hidden_to_hidden")

    in_to_hidden = self._in_to_hidden_linear(input_)
    hidden_to_hidden = self._hidden_to_hidden_linear(prev_state)
    output = self._activation(in_to_hidden + hidden_to_hidden)

    return output, output

  @property
  def state_size(self):
    return tf.TensorShape([self._hidden_size])

  @property
  def output_size(self):
    return tf.TensorShape([self._hidden_size])

In [ ]:
X = np.asarray(X, dtype="object")
y = np.array(y)

X = pad_sequences(X)

y = tf.keras.utils.to_categorical(y, num_classes=len(tokenizer.word_index) + 1)

##### Обучение модели

In [ ]:
model = Sequential()

model.add(Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=100))

model.add(LSTM(150, return_sequences=False))

model.add(Dense(len(tokenizer.word_index) + 1, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
history = model.fit(X, y, epochs=10, batch_size=64, validation_split=0.2)

Epoch 1/10
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 24s 17ms/step - accuracy: 0.0162 - loss: 8.1330 - val_accuracy: 0.0238 - val_loss: 7.9249
Epoch 2/10
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.0317 - loss: 7.5138 - val_accuracy: 0.0414 - val_loss: 7.8915
Epoch 3/10
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.0485 - loss: 7.2229 - val_accuracy: 0.0458 - val_loss: 7.8941
Epoch 4/10
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.0600 - loss: 6.8723 - val_accuracy: 0.0538 - val_loss: 7.9200
Epoch 5/10
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.0734 - loss: 6.4673 - val_accuracy: 0.0581 - val_loss: 8.0112
Epoch 6/10
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.0895 - loss: 6.0603 - val_accuracy: 0.0629 - val_loss: 8.1269
Epoch 7/10
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.1167 - loss: 5.6455 - val_accuracy: 0.0644 - val_loss: 8.2600
Epoch 8/10
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.1505 -

In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 49, 100)        │     1,142,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 11421)          │     1,724,571 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,051,815 (34.53 MB)

 Trainable params: 3,017,271 (11.51 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 6,034,544 (23.02 MB)

Тестирование

In [ ]:
def generate_text(seed_text, next_words, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

generated_text = generate_text("Due to Covid", 10, X.shape[1])
print(generated_text)

generated_text_2 = generate_text("If you lost your job", 5, X.shape[1])
print(generated_text_2)

generated_text_3 = generate_text("Don't panic", 7, X.shape[1])
print(generated_text_3)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
Due to Covid pandemic across u s consumer behavior voters participating democracy brands
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
If you lost your job money online shopping delivery services
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Don't panic bu